# 04 · Business analysis
Questions management asks, answered with SQL on the Gold layer.

In [ ]:
%pip install -q pyyaml python-dateutil

In [ ]:
# Setup: make the repo's `src` package importable and select the Databricks environment.
# The notebook runs from <repo>/notebooks inside a Databricks Git folder.
import os, sys
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ["LAKEHOUSE_ENV"] = "databricks"

dbutils.widgets.text("catalog", "workspace")          # Free Edition default; production: real_estate
catalog = dbutils.widgets.get("catalog")
overrides = {"environments": {"databricks": {"catalog": catalog,
                                             "raw_volume_root": f"/Volumes/{catalog}/raw/landing"}}}

from src.pipeline import build_lakehouse
lh, cfg = build_lakehouse(env="databricks", overrides=overrides)
print(f"catalog={cfg.catalog}  as_of={cfg.reporting_as_of}  batches={list(cfg.batches)}")

### Occupancy and value by property type

In [ ]:
display(spark.sql(f"""
SELECT property_type, COUNT(*) AS properties, SUM(total_units) AS units,
       ROUND(SUM(occupied_units) * 100.0 / SUM(total_units), 1) AS occupancy_pct,
       ROUND(SUM(current_value) / 1e6, 1) AS value_musd,
       ROUND(AVG(value_growth_pct), 1) AS avg_value_growth_pct
FROM {catalog}.gold.property_performance GROUP BY property_type ORDER BY value_musd DESC"""))

### Leases expiring in the next 90 days (renewal pipeline)

In [ ]:
display(spark.sql(f"""
SELECT property_name, COUNT(*) AS leases, ROUND(SUM(monthly_rent), 0) AS monthly_rent_at_risk
FROM {catalog}.gold.lease_expiry WHERE expires_within_90_days
GROUP BY property_name ORDER BY monthly_rent_at_risk DESC LIMIT 15"""))

### Delinquency by aging bucket

In [ ]:
display(spark.sql(f"""
SELECT aging_bucket, COUNT(DISTINCT tenant_id) AS tenants, ROUND(SUM(outstanding_balance), 0) AS balance
FROM {catalog}.gold.tenant_delinquency GROUP BY aging_bucket ORDER BY aging_bucket"""))

### Properties with collection rate below 90% this month

In [ ]:
display(spark.sql(f"""
SELECT property_name, property_type, collection_rate, outstanding_rent, occupancy_rate
FROM {catalog}.gold.property_performance WHERE collection_rate < 90 ORDER BY outstanding_rent DESC"""))

### Estimated NOI yield by property (assumption-based, see docs/architecture.md)

In [ ]:
display(spark.sql(f"""
SELECT property_name, property_type, ttm_collected_rent, ttm_maintenance_cost, estimated_noi,
       current_value, estimated_noi_yield_pct
FROM {catalog}.gold.property_performance ORDER BY estimated_noi_yield_pct DESC LIMIT 15"""))